In [1]:
import pandas as pd
import numpy as np

import joblib
import json

In [2]:
df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 151)


In [3]:
df["market_gap_score"] = (

      0.40 * df["market_opportunity_score"]

    + 0.35 * df["growth_score"]

    + 0.25 * (100 - df["competition_score"])

)

In [4]:
def gap_label(score):

    if score < 40:
        return "Low Opportunity"

    elif score < 70:
        return "Medium Opportunity"

    else:
        return "High Opportunity"


df["market_gap_label"] = (
    df["market_gap_score"]
    .apply(gap_label)
)

print(
    df["market_gap_label"]
    .value_counts()
)

market_gap_label
Medium Opportunity    44276
Low Opportunity        5664
High Opportunity         60
Name: count, dtype: int64


In [5]:
sector_gap = (

    df.groupby("sector")

    ["market_gap_score"]

    .mean()

    .sort_values(
        ascending=False
    )

)

print(
    sector_gap.head(20)
)

sector
HealthTech       49.572992
EdTech           49.543453
Cybersecurity    49.488795
AgriTech         49.470704
AI               49.363347
Retail           49.358596
E-Commerce       49.356020
TravelTech       49.354389
SaaS             49.340918
Gaming           49.336319
FoodTech         49.327263
PropTech         49.306644
Logistics        49.288704
CleanTech        49.249005
FinTech          49.158836
Name: market_gap_score, dtype: float64


In [6]:
top_gaps = (

    df[[
        "startup_name",
        "sector",
        "market_gap_score"
    ]]

    .sort_values(

        by="market_gap_score",

        ascending=False

    )

    .head(20)

)

print(top_gaps)

                     startup_name      sector  market_gap_score
46249   QuantumShift Technologies  E-Commerce         74.555241
20768              PathGo Digital    AgriTech         74.554988
30960         EduMatrix Analytics  HealthTech         74.209846
27570           LearnGo Solutions  TravelTech         74.138348
37173     MobileBridgeX Solutions      Gaming         73.785195
49052       BlockMatrix Analytics    PropTech         73.759440
38770          PropStack Ventures      EdTech         73.283282
14754  UltraScale Private Limited   CleanTech         72.826988
26506    LinkRise Private Limited  E-Commerce         72.619419
4390              HyperTechX Labs      Gaming         72.499837
11517            AppSmart Systems    FoodTech         72.434854
11091        HyperFusion Networks    AgriTech         72.297089
25787           ApexLoop Ventures      Gaming         72.121754
12631            FinPro Solutions      Gaming         71.937986
11173       EliteInsights Systems    Pro

In [7]:
def recommend_market_gap(sector):

    temp = df[
        df["sector"] == sector
    ]

    return {

        "sector": sector,

        "avg_gap_score":
            round(
                temp["market_gap_score"]
                .mean(),
                2
            ),

        "avg_growth":
            round(
                temp["growth_score"]
                .mean(),
                2
            ),

        "avg_opportunity":
            round(
                temp["market_opportunity_score"]
                .mean(),
                2
            )
    }

In [8]:
print(
    recommend_market_gap(
        "AI"
    )
)

print(
    recommend_market_gap(
        "FinTech"
    )
)

print(
    recommend_market_gap(
        "HealthTech"
    )
)

{'sector': 'AI', 'avg_gap_score': 49.36, 'avg_growth': 52.36, 'avg_opportunity': 21.93}
{'sector': 'FinTech', 'avg_gap_score': 49.16, 'avg_growth': 52.11, 'avg_opportunity': 21.9}
{'sector': 'HealthTech', 'avg_gap_score': 49.57, 'avg_growth': 52.8, 'avg_opportunity': 22.06}


In [10]:
engine = {

    "engine_name":
        "Market Gap Detection Engine",

    "version":
        "V1"
}

joblib.dump(

    engine,

    "../models/market_gap_detection_engine/market_gap_detection_engine.pkl"
)

['../models/market_gap_detection_engine/market_gap_detection_engine.pkl']

In [11]:
metadata = {

    "engine":
        "Market Gap Detection",

    "output":

    [

        "market_gap_score",

        "market_gap_label"

    ]
}

with open(

    "../models/market_gap_detection_engine/metadata.json",

    "w"

) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )

In [12]:
df.to_csv(
    "../datasets/cleaned/startup_info.csv",
    index=False
)

print(
    "Market Gap Engine Complete ✅"
)

Market Gap Engine Complete ✅


In [13]:
print(df["market_gap_score"].describe())

print(df["market_gap_label"].value_counts())

print(sector_gap.head(10))

count    50000.000000
mean        49.368035
std          7.636729
min         16.766075
25%         44.036264
50%         49.383945
75%         54.762422
max         74.555241
Name: market_gap_score, dtype: float64
market_gap_label
Medium Opportunity    44276
Low Opportunity        5664
High Opportunity         60
Name: count, dtype: int64
sector
HealthTech       49.572992
EdTech           49.543453
Cybersecurity    49.488795
AgriTech         49.470704
AI               49.363347
Retail           49.358596
E-Commerce       49.356020
TravelTech       49.354389
SaaS             49.340918
Gaming           49.336319
Name: market_gap_score, dtype: float64
